# 第07章 信任的边界 · A组实验


## 先看一幕：地图之外，还能相信吗？

机器人在熟悉的街区导航得很准。一天，它走出地图覆盖范围，仍然自信地延长路线：熟悉区域里的好表现，究竟能支撑多远的信任？

用 $[-1,1]$ 内的数据拟合 $\sin(x)$ 后，为什么仍需单独评估 $[2,3]$ 上的误差？训练误差小，缺少了哪些条件才谈得上域外保证？

地图是训练域与测试域的比喻，图中的路线不代表测量曲线。本文实验是确定性函数外推；统计置信、模型校准与理论误差界需要各自的假设和证据，不能用同一张图互相替代。

**知识主题：** 泛化、误差界与可信性

**本章核心问题：** 当AI给出一个答案时，我们凭什么相信它？

**核心概念：** 泛化界、分布外泛化、误差界

**先修知识：** 概率基础、函数逼近、误差度量与训练/测试划分

**课时：** 4节 × 2学时 = 8学时；全书八章共64学时。

### 本章四节安排

| 节 | 内容 | 学时 |
| --- | --- | --- |
| 7.1 | 逼近理论与泛化界 | 2 |
| 7.2 | 分布内与分布外泛化 | 2 |
| 7.3 | 机器学习与降阶模型的共性 | 2 |
| 7.4 | AI关联：不确定性量化 | 2 |

```python
import sys
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)
print("Python:", sys.version.split()[0], "NumPy:", np.__version__)
```

## 7.1 逼近理论与泛化界（2学时）

**学习目标**　能区分表示能力、训练误差与对未见样本的保证。

取有限假设类与有界损失，比较样本数增大时的经验风险、独立测试误差和理论界。界使用了哪些独立性与置信水平假设？

**实验设计。** 取一维输入 $x\in[0,1]$，真实标签 $y=\mathbb{1}[x\ge 0.5]$ 并以概率 $\eta=0.1$ 翻转。假设类取 $11$ 个阈值分类器 $\mathcal{H}=\{x\mapsto\mathbb{1}[x\ge \theta]:\theta\in\{0,0.1,\dots,1.0\}\}$。对每个样本量 $n$，独立采样训练集，选择经验风险最小的假设 $h_n$，计算其经验风险 $\hat{L}(h_n)$ 与在固定大测试集上的真实风险 $L(h_n)$。有限类 Hoeffding 界为
$$
L(h)\le \hat{L}(h)+\sqrt{\frac{\ln(|\mathcal{H}|/\delta)}{2n}},
$$
取 $\delta=0.05$。重复 $T=200$ 次，记录真实风险超过界的频率。

```python
# 7.1 有限假设类的泛化界实验

def sample_threshold_data(n, rng, noise=0.1):
    x = rng.uniform(0, 1, n)
    y_clean = (x >= 0.5).astype(int)
    flip = rng.random(n) < noise
    y = np.where(flip, 1 - y_clean, y_clean)
    return x, y

thresholds = np.linspace(0, 1, 11)
H = [lambda x, t=t: (x >= t).astype(int) for t in thresholds]

# 固定大测试集
x_test, y_test = sample_threshold_data(200_000, rng)

def empirical_risk(h, x, y):
    return np.mean(h(x) != y)

def true_risk(h):
    return np.mean(h(x_test) != y_test)

n_values = [10, 20, 50, 100, 200, 500, 1000]
T = 200
delta = 0.05
log_term = np.log(len(H) / delta)

records = []
for n in n_values:
    emp_list, true_list, bound_list = [], [], []
    violations = 0
    for _ in range(T):
        x_tr, y_tr = sample_threshold_data(n, rng)
        risks = [empirical_risk(h, x_tr, y_tr) for h in H]
        best = int(np.argmin(risks))
        h_best = H[best]
        emp = risks[best]
        true = true_risk(h_best)
        bound = emp + np.sqrt(log_term / (2 * n))
        emp_list.append(emp)
        true_list.append(true)
        bound_list.append(bound)
        if true > bound:
            violations += 1
    records.append({
        "n": n,
        "emp_mean": np.mean(emp_list),
        "true_mean": np.mean(true_list),
        "true_std": np.std(true_list),
        "bound_mean": np.mean(bound_list),
        "violation_rate": violations / T,
    })

print(f"{'n':>6} {'emp_risk':>10} {'true_risk':>10} {'true_std':>9} {'bound':>9} {'viol_rate':>10}")
for r in records:
    print(f"{r['n']:>6} {r['emp_mean']:>10.4f} {r['true_mean']:>10.4f} "
          f"{r['true_std']:>9.4f} {r['bound_mean']:>9.4f} {r['violation_rate']:>10.3f}")

# 作图
ns = [r["n"] for r in records]
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(ns, [r["emp_mean"] for r in records], "o-", label="经验风险（训练误差）")
ax.plot(ns, [r["true_mean"] for r in records], "s-", label="真实风险（测试误差）")
ax.plot(ns, [r["bound_mean"] for r in records], "^--", label="有限类 Hoeffding 界（均值）")
ax.set_xscale("log")
ax.set_xlabel("样本量 $n$")
ax.set_ylabel("风险")
ax.set_title("7.1 经验风险、真实风险与泛化界随样本量的变化")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
```

### 结果与讨论

预期观察：（1）经验风险与真实风险均随 $n$ 增大而趋近噪声率 $0.1$；（2）小样本下真实风险可能显著高于经验风险（过拟合）；（3）理论界随 $n$ 增大以 $O(n^{-1/2})$ 速率收紧；（4）真实风险超过界的频率应接近或低于 $\delta=0.05$。

**讨论要点：** 界成立依赖三个条件——样本独立同分布、损失有界于 $[0,1]$、假设类有限且事先固定。VC 维界将有限类推广到无限类，但以 VC 维 $d$ 替代 $\ln|\mathcal{H}|$，通常更松。Rademacher 复杂度进一步利用数据分布信息。三者共同回答“训练误差小是否足以保证测试误差小”，答案是否定的：还需要假设空间复杂度与样本量共同约束。

## 7.2 分布内与分布外泛化（2学时）

**学习目标**　能按科学任务划分数据，并建立独立的分布外测试。

在 $[0,4]$ 上用多项式拟合 $\sin x$，分别在训练节点、区间内的新点及 $[4.5,6.0]$、$[7.0,9.0]$ 的新点上测量误差。改变训练与测试区间会出现什么差别？

**实验设计（严格按正文 7.2.5）。** 真实函数 $f(x)=\sin x$，噪声 $\sigma=0.1$。训练区间 $[0,4]$，$m=50$ 个均匀采样带噪点。评价区间：内插区 $[0.5,3.5]$、邻近外推区 $[4.5,6.0]$、远距外推区 $[7.0,9.0]$，各取 $N=200$ 个等距点。模型为三次与五次多项式（最小二乘拟合）。重复 $T=100$ 次，报告 RMSE 均值与标准差。基线为训练区间上的常数预测 $h_0(x)=\bar y_{\text{train}}$。

```python
# 7.2 多项式回归的内插与外推

from numpy.polynomial import Polynomial

f = np.sin
sigma = 0.1
train_lo, train_hi = 0.0, 4.0
m = 50
N_eval = 200
T = 100

eval_intervals = {
    "内插区 [0.5,3.5]": (0.5, 3.5),
    "邻近外推 [4.5,6.0]": (4.5, 6.0),
    "远距外推 [7.0,9.0]": (7.0, 9.0),
}
degrees = [3, 5]

def rmse(y_pred, y_true):
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

results = {deg: {k: [] for k in eval_intervals} for deg in degrees}
baseline_results = {k: [] for k in eval_intervals}

for _ in range(T):
    x_train = rng.uniform(train_lo, train_hi, m)
    y_train = f(x_train) + rng.normal(0, sigma, m)
    y_bar = np.mean(y_train)
    for name, (lo, hi) in eval_intervals.items():
        x_eval = np.linspace(lo, hi, N_eval)
        y_true = f(x_eval)
        baseline_results[name].append(rmse(np.full_like(y_true, y_bar), y_true))
        for deg in degrees:
            model = Polynomial.fit(x_train, y_train, deg=deg).convert()
            results[deg][name].append(rmse(model(x_eval), y_true))

print(f"{'区间':<22} {'三次均值':>10} {'三次标准差':>10} {'五次均值':>10} {'五次标准差':>10} {'基线均值':>10}")
for name in eval_intervals:
    r3 = np.array(results[3][name])
    r5 = np.array(results[5][name])
    rb = np.array(baseline_results[name])
    print(f"{name:<22} {r3.mean():>10.4f} {r3.std():>10.4f} {r5.mean():>10.4f} {r5.std():>10.4f} {rb.mean():>10.4f}")

# 作图
fig, ax = plt.subplots(figsize=(7, 4.2))
x_plot = np.linspace(0, 9, 600)
ax.plot(x_plot, f(x_plot), "k-", lw=2, label="真实函数 $\\sin x$")
for deg, style in zip(degrees, ["C0--", "C1-."]):
    # 用固定种子的一次代表性拟合
    rng_rep = np.random.default_rng(2026)
    x_tr = rng_rep.uniform(train_lo, train_hi, m)
    y_tr = f(x_tr) + rng_rep.normal(0, sigma, m)
    mdl = Polynomial.fit(x_tr, y_tr, deg=deg).convert()
    ax.plot(x_plot, mdl(x_plot), style, label=f"{deg} 次多项式")
ax.axvspan(train_lo, train_hi, color="tab:green", alpha=0.08, label="训练区间")
ax.axvspan(4.5, 6.0, color="tab:orange", alpha=0.08, label="邻近外推")
ax.axvspan(7.0, 9.0, color="tab:red", alpha=0.08, label="远距外推")
ax.set_ylim(-2, 2)
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.set_title("7.2 三次与五次多项式在训练区间内外的行为")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()
```

### 结果与讨论

预期观察：（1）内插区 RMSE 接近或略高于噪声水平 $0.1$，三次与五次差异不显著；（2）邻近外推区误差明显上升，五次多项式方差更大；（3）远距外推区五次多项式 RMSE 显著大于三次，且随外推距离急剧增大；（4）常数基线在三个区间误差均在 $0.7$–$0.8$ 量级，指示“无外推能力”的误差尺度。

**讨论要点：** 划分方案的合法性取决于所声称的评价目标。若目标是“参数内插”，按参数分组划分可接受，但需注明测试参数位于训练范围内；若目标是“参数外推”，则必须采用训练与测试参数区间不相交的设计。随机按数据点划分在时序或参数化问题中会造成信息泄漏——同一参数下的时间点高度相关，测试误差反映的是插值而非泛化。7.2.5 节末尾的随堂提问（方案 A/B/C）正是这一原则的操作化检验。

## 7.3 机器学习与降阶模型的共性：误差界与 V&V（2学时）

**学习目标**　能用一个统一的误差分解比较 ROM 与机器学习代理模型，并区分 Verification 与 Validation 的证据层级。

**实验设计（按正文 7.3.4）。** 一维参数化椭圆方程 $-\partial_{xx}u(x;\mu)=\mu^2\sin(\pi x)+2\mu$，$x\in[0,1]$，$u(0)=u(1)=0$，精确解 $u(x;\mu)=\frac{\mu^2}{\pi^2}\sin(\pi x)+\mu x(1-x)$。FOM 取有限差分，$N=200$。**ROM 分支**：训练参数 $\mu\in\{0.1,\dots,0.5\}$ 上计算 FOM 快照，POD 提取 $r=3$ 个基，Galerkin 投影。**MLP 分支**：相同训练数据，输入 $(\mu,x)$、输出 $u$，隐藏层宽度 $64$、tanh、Adam、固定种子。评价：分布内 $\mu\in\{0.15,0.25,0.35,0.45\}$、边界邻近 $\mu\in[0.55,0.65]$、远距外推 $\mu\in[0.75,1.0]$。重复 $T=20$，报告相对 $L^2$ 误差均值与标准差。

```python
# 7.3 ROM 与 MLP 代理的误差分解对比
import torch
import torch.nn as nn

def fem_solve_interior(mu, N=200):
    """返回内部节点、FOM 解、刚度矩阵与右端项。"""
    h = 1.0 / (N + 1)
    x_int = np.linspace(0, 1, N + 2)[1:-1]
    main = 2.0 / h**2 * np.ones(N)
    off = -1.0 / h**2 * np.ones(N - 1)
    A = np.diag(main) + np.diag(off, 1) + np.diag(off, -1)
    g = mu**2 * np.sin(np.pi * x_int) + 2 * mu
    u_int = np.linalg.solve(A, g)
    return x_int, u_int, A, g

def exact_solution(x, mu):
    return mu**2 / np.pi**2 * np.sin(np.pi * x) + mu * x * (1 - x)

N_fom = 200
x_int = np.linspace(0, 1, N_fom + 2)[1:-1]
mu_train = np.array([0.1, 0.2, 0.3, 0.4, 0.5])

# --- ROM 分支 ---
snapshots = []
for mu in mu_train:
    _, u_int, _, _ = fem_solve_interior(mu, N_fom)
    snapshots.append(u_int)
U = np.column_stack(snapshots)
Phi, S, _ = np.linalg.svd(U, full_matrices=False)
r = 3
Phi_r = Phi[:, :r]

def rom_predict(mu):
    _, _, A, g = fem_solve_interior(mu, N_fom)
    A_r = Phi_r.T @ A @ Phi_r
    g_r = Phi_r.T @ g
    u_hat = np.linalg.solve(A_r, g_r)
    return Phi_r @ u_hat

# --- MLP 分支 ---
X_list, Y_list = [], []
for mu in mu_train:
    _, u_int, _, _ = fem_solve_interior(mu, N_fom)
    for xi, ui in zip(x_int, u_int):
        X_list.append([mu, xi])
        Y_list.append(ui)
X_t = torch.tensor(X_list, dtype=torch.float32)
Y_t = torch.tensor(Y_list, dtype=torch.float32).unsqueeze(1)

class MLP(nn.Module):
    def __init__(self, width=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, width), nn.Tanh(),
            nn.Linear(width, width), nn.Tanh(),
            nn.Linear(width, 1),
        )
    def forward(self, x):
        return self.net(x)

def train_mlp(seed):
    torch.manual_seed(seed)
    mlp = MLP()
    opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    for _ in range(1000):
        opt.zero_grad()
        loss = loss_fn(mlp(X_t), Y_t)
        loss.backward()
        opt.step()
    return mlp

def relative_l2(pred, ref):
    return float(np.linalg.norm(pred - ref) / np.linalg.norm(ref))

mu_in = [0.15, 0.25, 0.35, 0.45]
mu_near = [0.55, 0.60, 0.65]
mu_far = [0.75, 0.85, 0.95, 1.0]
groups = {"分布内": mu_in, "边界邻近": mu_near, "远距外推": mu_far}

T_rep = 20
rom_errs = {k: [] for k in groups}
mlp_errs = {k: [] for k in groups}

for t in range(T_rep):
    for gname, mus in groups.items():
        for mu in mus:
            u_ref = exact_solution(x_int, mu)
            rom_errs[gname].append(relative_l2(rom_predict(mu), u_ref))
    mlp = train_mlp(seed=2026 + t)
    with torch.no_grad():
        for gname, mus in groups.items():
            for mu in mus:
                inp = torch.tensor(np.column_stack([np.full_like(x_int, mu), x_int]),
                                   dtype=torch.float32)
                u_pred = mlp(inp).squeeze().numpy()
                u_ref = exact_solution(x_int, mu)
                mlp_errs[gname].append(relative_l2(u_pred, u_ref))

print(f"{'区间':<10} {'ROM 均值':>10} {'ROM 标准差':>11} {'MLP 均值':>10} {'MLP 标准差':>11}")
for gname in groups:
    r = np.array(rom_errs[gname]); m_ = np.array(mlp_errs[gname])
    print(f"{gname:<10} {r.mean():>10.2e} {r.std():>11.2e} {m_.mean():>10.2e} {m_.std():>11.2e}")

# 作图
fig, ax = plt.subplots(figsize=(7, 4.2))
labels = list(groups.keys())
x_pos = np.arange(len(labels))
width = 0.35
rom_means = [np.mean(rom_errs[k]) for k in labels]
rom_stds = [np.std(rom_errs[k]) for k in labels]
mlp_means = [np.mean(mlp_errs[k]) for k in labels]
mlp_stds = [np.std(mlp_errs[k]) for k in labels]
ax.bar(x_pos - width/2, rom_means, width, yerr=rom_stds, capsize=4, label="ROM (POD-Galerkin)")
ax.bar(x_pos + width/2, mlp_means, width, yerr=mlp_stds, capsize=4, label="MLP 代理")
ax.set_yscale("log")
ax.set_xticks(x_pos)
ax.set_xticklabels(labels)
ax.set_ylabel("相对 $L^2$ 误差")
ax.set_title("7.3 ROM 与 MLP 在分布内、边界邻近与远距外推区的误差")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
```

### 结果与讨论

预期观察：（1）分布内区两者误差均接近 FOM 离散地板（$10^{-4}$ 量级）；（2）边界邻近区误差上升，MLP 上升幅度可能大于 ROM，因 ROM 的全局基具有参数插值能力；（3）远距外推区两者误差均显著增大，ROM 的表示误差（POD 截断）可能成为主导，MLP 的误差难以分离表示与拟合成分。

**讨论要点（V&V 证据层级）：** 以 POD-Galerkin ROM 为例，三组证据分属不同层级。（A）$\mu\in[0.1,0.5]$ 内 ROM 与 FOM 快照的相对误差 $10^{-4}$——这是**解验证**，比较对象是同一数学模型的离散解，不构成 Validation；（B）$\mu\in[0.6,1.0]$ 内 ROM 与高保真 DNS 的相对误差 $10^{-2}$——这是 **Validation 的候选证据**，但需检查 DNS 是否独立于 ROM 构造、其自身不确定度是否量化、参数范围是否覆盖目标用途；（C）$\mu\in[0.1,0.5]$ 内 FOM 网格加密检验显示 $O(h^2)$ 收敛——这是**代码验证与 FOM 解验证**，是（A）与（B）中所有比较的前提。三组证据不能相互替代：代码验证通过不蕴含模型验证通过，分布内精度不构成分布外有效性的证据。

## 7.4 AI关联：不确定性量化（2学时）

**学习目标**　能评价不确定性表达是否与实际误差和覆盖表现一致。

为小规模回归问题构造预测区间，分别在同分布和域外测试集中比较覆盖率与平均宽度。相同覆盖率是否意味着预测同样有用？

**实验设计（按正文 7.4.3）。** 真实函数 $f(x)=\sin x$，噪声 $\sigma=0.1$。基础模型为三次多项式回归。非一致性分数取绝对残差。训练集 $n_{\text{train}}=100$，$x\in[0,2]$；校准集 $n_{\text{cal}}\in\{50,200,1000\}$，$x\in[0,2]$；分布内测试集 $n_{\text{test}}=2000$，$x\in[0,2]$；分布外测试集 $n_{\text{test}}=2000$，$x\in[3,4]$。标称覆盖率 $1-\alpha\in\{0.80,0.90,0.95\}$。重复 $T=100$，每次重新拟合模型、重新计算分位数。基线为固定宽度区间 $\hat\mu(x)\pm 1.96\hat\sigma$。

```python
# 7.4 分裂共形预测的覆盖率与宽度

from numpy.polynomial import Polynomial

f = np.sin
sigma = 0.1
n_train = 100
n_cal_values = [50, 200, 1000]
alphas = [0.20, 0.10, 0.05]
n_test = 2000
T = 100

def gen(n, lo, hi, rng):
    x = rng.uniform(lo, hi, n)
    y = f(x) + rng.normal(0, sigma, n)
    return x, y

# 固定测试集，保证跨重复可比
x_in, y_in = gen(n_test, 0, 2, rng)
x_out, y_out = gen(n_test, 3, 4, rng)

results = {}
for n_cal in n_cal_values:
    for alpha in alphas:
        cov_in, cov_out, wid_in, wid_out = [], [], [], []
        for _ in range(T):
            x_tr, y_tr = gen(n_train, 0, 2, rng)
            model = Polynomial.fit(x_tr, y_tr, deg=3).convert()
            x_cal, y_cal = gen(n_cal, 0, 2, rng)
            scores = np.abs(y_cal - model(x_cal))
            k = int(np.ceil((n_cal + 1) * (1 - alpha)))
            q = np.sort(scores)[k - 1] if k <= n_cal else np.inf
            for x_te, y_te, cov, wid in [
                (x_in, y_in, cov_in, wid_in),
                (x_out, y_out, cov_out, wid_out),
            ]:
                pred = model(x_te)
                cov.append(np.mean(np.abs(y_te - pred) <= q))
                wid.append(2 * q)
        results[(n_cal, alpha)] = {
            "cov_in": (np.mean(cov_in), np.std(cov_in)),
            "cov_out": (np.mean(cov_out), np.std(cov_out)),
            "wid_in": (np.mean(wid_in), np.std(wid_in)),
            "wid_out": (np.mean(wid_out), np.std(wid_out)),
        }

print(f"{'n_cal':>6} {'1-alpha':>8} {'cov_in':>8} {'cov_out':>8} {'width_in':>9} {'width_out':>10}")
for (n_cal, alpha), r in results.items():
    print(f"{n_cal:>6} {1-alpha:>8.2f} {r['cov_in'][0]:>8.3f} {r['cov_out'][0]:>8.3f} "
          f"{r['wid_in'][0]:>9.3f} {r['wid_out'][0]:>10.3f}")

# 作图
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax = axes[0]
for n_cal, marker in zip(n_cal_values, ["o", "s", "^"]):
    covs = [results[(n_cal, a)]["cov_in"][0] for a in alphas]
    ax.plot([1-a for a in alphas], covs, marker=marker, label=f"n_cal={n_cal}（域内）")
    covs_out = [results[(n_cal, a)]["cov_out"][0] for a in alphas]
    ax.plot([1-a for a in alphas], covs_out, marker=marker, ls="--", alpha=0.6,
            label=f"n_cal={n_cal}（域外）")
ax.plot([0.75, 1.0], [0.75, 1.0], "k:", lw=1, label="标称=实测")
ax.set_xlabel("标称覆盖率 $1-\\alpha$")
ax.set_ylabel("实测覆盖率")
ax.set_title("覆盖率：域内 vs 域外")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

ax = axes[1]
for n_cal, marker in zip(n_cal_values, ["o", "s", "^"]):
    widths = [results[(n_cal, a)]["wid_in"][0] for a in alphas]
    ax.plot([1-a for a in alphas], widths, marker=marker, label=f"n_cal={n_cal}")
ax.set_xlabel("标称覆盖率 $1-\\alpha$")
ax.set_ylabel("平均区间宽度")
ax.set_title("区间宽度随标称覆盖率增大")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
```

### 结果与讨论

预期观察：（1）分布内实测覆盖率不低于标称值，超出幅度不超过 $1/(n_{\text{cal}}+1)$，且随 $n_{\text{cal}}$ 增大更接近标称值；（2）分布外覆盖率显著低于标称值，且 $n_{\text{cal}}$ 增大不能修复该偏差；（3）区间宽度随标称覆盖率提高而单调增大，$n_{\text{cal}}$ 较小时宽度更大（经验分位数更保守）。

**讨论要点：** 共形预测的覆盖保证是**边际的**而非条件的：$\mathbb{P}(Y\in C_\alpha(X))\ge 1-\alpha$ 中的概率同时取于校准样本与测试点，不条件于 $X=x$。对某些 $x$，条件覆盖率可能显著低于标称水平。Barber 等证明，在无额外假设的有限样本下，精确的条件覆盖不可能对任意分布实现。此外，保证依赖**可交换性**：当测试输入落在校准支撑之外时，可交换性被破坏，覆盖率无理论下界。因此，报告预测区间时必须同时报告标称覆盖率、实测覆盖率、区间宽度与评价分布，四者缺一不可。两模型覆盖率相近（如 $0.901$ 与 $0.903$，差异远小于抽样标准误）时，宽度更小者信息量更高，但该优势必须结合分布外行为与部署场景共同评估。

## 本章结论

本章围绕“信任的边界”建立了三层递进的证据框架。**7.1** 表明经验风险小不蕴含泛化误差小：有限类 Hoeffding 界、VC 维界与 Rademacher 复杂度分别从不同角度量化了假设空间复杂度与样本量对泛化误差的约束，而近似-估计分解揭示了容量选择的根本张力。**7.2** 将评价从分布内推广到分布外：协变量偏移下的 Ben-David 界说明目标域风险由源域风险、分布差异与理想联合风险共同控制，而多项式外推实验展示了当评价区间超出训练覆盖时，误差不再受样本密度控制。数据划分的合法性取决于所声称的评价目标——参数外推必须采用区间不相交设计。**7.3** 将机器学习代理与降阶模型纳入统一的误差分解：表示误差界定上限，拟合误差可通过数据与优化减小，V&V 层级要求任何精度声明必须说明其在哪个参数区域、相对于哪个参考、以何种范数成立。**7.4** 引入分裂共形预测：在可交换性下提供有限样本边际覆盖保证，但该保证随分布偏移失效，且边际覆盖不蕴含条件覆盖。四节共同回答本章核心问题：AI 的答案之所以可信，不是因为算得快或模型大，而是因为我们清楚地知道它**在哪里学的、条件变没变、能用什么证据去检验**。

## 参考资料

| 内容 | 作者与出处 | 用途 |
| --- | --- | --- |
| 有限类 Hoeffding 界与 VC 界 | Shalev-Shwartz & Ben-David, *Understanding Machine Learning*, Cambridge, 2014, Ch. 4–6 | 7.1 理论依据 |
| Rademacher 复杂度 | Bartlett & Mendelson, *JMLR*, 2002 | 7.1 讨论 |
| 协变量偏移下的目标风险界 | Ben-David et al., *Machine Learning*, 79:151–175, 2010 | 7.2 理论依据 |
| 重要性加权与直接重要性估计 | Sugiyama et al., *JMLR*, 8:985–1005, 2007; *AISM*, 60:699–746, 2008 | 7.2 讨论 |
| 多项式与样条外推的数值对比 | Puiu & Puiu, *Sci. Bull. Politeh. Univ. Bucharest*, 2024 | 7.2 实验参照 |
| Operator Inference | Peherstorfer & Willcox, *CMAME*, 306:196–215, 2016 | 7.3 ROM 构造 |
| DWR 与 ML 误差估计对比 | Blonigan & Parish, *CMAME*, 409:115988, 2023 | 7.3 误差估计讨论 |
| V&V 经典框架 | Oberkampf & Roy, *Verification and Validation in Scientific Computing*, Cambridge, 2010, Ch. 1–2 | 7.3 V&V 层级 |
| 共形预测奠基工作 | Vovk, Gammerman & Shafer, *Algorithmic Learning in a Random World*, Springer, 2005/2022 | 7.4 理论依据 |
| 分裂共形预测教程 | Angelopoulos & Bates, arXiv:2107.07511, 2021 | 7.4 定理陈述 |
| 条件覆盖的不可能性 | Barber et al., *Information and Inference*, 10(2):455–482, 2021 | 7.4 讨论 |
| 分布偏移下的共形预测 | Tibshirani et al., *NeurIPS*, 32, 2019 | 7.4 讨论 |